# Positional encoding
Positional encoding is a technique used in Transformer models (and some other sequence-based architectures) to inject information about the position of tokens in a sequence. Since Transformers process sequences in parallel and don't have inherent order (unlike RNNs), positional encodings help the model understand the sequential structure and relative positions of elements.

### Why Is It Needed?
- Transformers rely on self-attention, which is permutation-invariant (i.e., it doesn't care about order). Without positional information, "The cat sat" and "sat cat The" would be treated identically.
- Positional encodings add a signal that varies with position, allowing the model to learn order-dependent relationships.

### How It Works
- **Input**: Each token in the sequence gets a positional vector added to its embedding.
- **Common Method (Sinusoidal Encoding)**: Used in the original Transformer paper. For position \( pos \) and dimension \( i \):
  - Even indices: \( PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right) \)
  - Odd indices: \( PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right) \)
  - This creates unique patterns for each position, with different frequencies for different dimensions.
- **Learned Positional Encodings**: Some models (e.g., BERT) learn positional embeddings instead of using fixed functions.
- **Properties**:
  - Relative positioning: The encoding allows the model to compute relative distances (e.g., \( PE(pos+k) \) can be expressed using \( PE(pos) \)).
  - Fixed size: Handles sequences up to a maximum length (e.g., 512 in BERT).

### In Transformers
- Added to input embeddings: \( \text{Input} = \text{Embedding}(token) + PE(position) \)
- Passed through the model like any other feature.

### Example in Code 
```python
import tensorflow as tf
from tensorflow.keras.layers import Layer
import numpy as np

class PositionalEncoding(Layer):
    def __init__(self, d_model, max_len=1000, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.max_len = max_len

        position = np.arange(max_len)[:, np.newaxis]
        i = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000.0, (2 * (i // 2)) / np.float32(d_model))
        angle_rads = position * angle_rates

        pos_encoding = np.zeros((max_len, d_model))
        pos_encoding[:, 0::2] = np.sin(angle_rads[:, 0::2])
        pos_encoding[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.pos_encoding = tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pos_encoding[:, :seq_len, :]

# Example usage
vocab_size = 40
seq_len = 12
embedding_dim = 64

sample_tokens = tf.random.uniform((2, seq_len), minval=0, maxval=vocab_size, dtype=tf.int32)
embed_layer = tf.keras.layers.Embedding(vocab_size, embedding_dim)
embedded = embed_layer(sample_tokens)

pos_layer = PositionalEncoding(embedding_dim, max_len=100)
encoded = pos_layer(embedded)

print('Embedding shape:', embedded.shape)
print('After positional encoding:', encoded.shape)```

